In [ ]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import numpy as np
import math
import time
import os

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise SystemExit("Error: Could not open video.")

detector = HandDetector(maxHands=1)
offset = 20
imgSize = 300
counter = 0
folder = "Data/train/CLASS_1"
os.makedirs(folder, exist_ok=True)

while True:
    success, img = cap.read()
    if not success:
        print("Error: Could not read frame.")
        break

    hands, img = detector.findHands(img)
    imgWhite = None

    if hands:
        hand = hands[0]
        x, y, w, h = hand["bbox"]
        imgWhite = np.ones((imgSize, imgSize, 3), np.uint8) * 255

        x_start = max(0, x - offset)
        y_start = max(0, y - offset)
        x_end = min(img.shape[1], x + w + offset)
        y_end = min(img.shape[0], y + h + offset)
        imgCrop = img[y_start:y_end, x_start:x_end]

        if imgCrop.size == 0:
            continue

        aspect_ratio = h / w
        if aspect_ratio > 1:
            k = imgSize / h
            w_cal = math.ceil(k * w)
            img_resize = cv2.resize(imgCrop, (w_cal, imgSize))
            w_gap = math.ceil((imgSize - w_cal) / 2)
            imgWhite[:, w_gap : w_cal + w_gap] = img_resize
        else:
            k = imgSize / w
            h_cal = min(math.ceil(k * h), imgSize)
            img_resize = cv2.resize(imgCrop, (imgSize, h_cal))
            h_gap = math.ceil((imgSize - h_cal) / 2)
            imgWhite[h_gap : h_cal + h_gap, :] = img_resize

        cv2.imshow("ImageCrop", imgCrop)
        cv2.imshow("ImageWhite", imgWhite)

    cv2.imshow("Image", img)
    key = cv2.waitKey(1)

    if key == ord("s") and imgWhite is not None:
        counter += 1
        cv2.imwrite(f"{folder}/Image_{time.time()}.jpg", imgWhite)
        print(counter)
    elif key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
